# Hybrid OLS + Year-on-Year degradation example

This notebook demonstrates `rdtools.degradation_hybrid_ols_yoy`, which splits a normalized-energy time series at the one-year mark and fits:

- **Ordinary least squares** on year 1, producing a rate in %/year of the year-0 system capacity, and
- **Year-on-year** on the remainder, producing a steady-state rate in %/year of the start-of-year-2 capacity.

The two-piece fit is useful when the first year of operation degrades qualitatively differently from steady state (light-induced degradation, light-soaking, initial stabilization). A single-rate fit blends those regimes and hides the nonlinearity.

The function is also exposed through `TrendAnalysis` as the `"hybrid_degradation"` analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rdtools
%matplotlib inline

## 1. Build a synthetic two-rate series

Five years of daily normalized energy:
- Year 1 degrades at -2.0 %/year (e.g. an early-life loss),
- Years 2-5 degrade at -0.5 %/year (steady state).

A small annual oscillation and Gaussian noise are added so the series resembles real data.

In [ ]:
np.random.seed(0)

idx = pd.date_range('2018-01-01', '2023-01-01', freq='D')
years = (idx - idx[0]) / pd.Timedelta('365D')

rd_year1, rd_year2plus = -2.0, -0.5  # %/year
y0 = 1.0
y_end_year1 = y0 + (rd_year1 / 100) * 1.0

true_trend = np.where(
    years < 1,
    y0 + (rd_year1 / 100) * years,
    y_end_year1 + (rd_year2plus / 100) * (years - 1),
)
seasonal = 0.02 * np.cos(2 * np.pi * years)
noise = np.random.normal(0, 0.01, len(idx))

energy_normalized = pd.Series(true_trend + seasonal + noise, index=idx)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(idx, energy_normalized, '.', alpha=0.25, label='noisy series')
ax.plot(idx, true_trend, 'k-', lw=2, label='true two-rate trend')
ax.axvline(idx[0] + pd.DateOffset(years=1), color='red', ls='--', lw=1,
           label='year-1 / year-2+ split')
ax.set_ylabel('Normalized energy')
ax.legend(loc='lower left')
fig.autofmt_xdate()
plt.show()

## 2. Compare single-rate methods to the hybrid method

Run OLS and year-on-year on the **whole** series, then run the hybrid method, and compare each to the true rates.

In [ ]:
ols_rd, ols_ci, _ = rdtools.degradation_ols(energy_normalized)
yoy_rd, yoy_ci, _ = rdtools.degradation_year_on_year(energy_normalized)

rd1_hat, rd2_hat, hybrid_info = rdtools.degradation_hybrid_ols_yoy(
    energy_normalized
)

rows = [
    ('OLS, whole series',          ols_rd,  ols_ci),
    ('YoY, whole series',          yoy_rd,  yoy_ci),
    ('Hybrid, year 1 (OLS)',       rd1_hat, hybrid_info['year1'][1]),
    ('Hybrid, years 2+ (YoY)',     rd2_hat, hybrid_info['years2plus'][1]),
]
comparison = pd.DataFrame(
    [(name, f'{rd:+.3f}', f'[{ci[0]:+.3f}, {ci[1]:+.3f}]')
     for name, rd, ci in rows],
    columns=['method', 'Rd (%/yr)', '68% CI']
)
print(f'True rates: year 1 = {rd_year1:+.2f} %/yr, '
      f'years 2+ = {rd_year2plus:+.2f} %/yr')
comparison

Notice that the single-rate methods report something close to the *time-weighted average* of the two true rates and therefore underestimate the year-1 loss and overestimate the steady-state loss. The hybrid method recovers each regime separately.

## 3. Visualizing the two fits

Plot the OLS line on the year-1 window and the YoY median trend on the year-2+ window.

In [ ]:
split = hybrid_info['split_date']

# Year-1 OLS line (in its own normalized units)
year1_info = hybrid_info['year1'][2]
year1_idx = energy_normalized.loc[:split].index
year1_years = (year1_idx - year1_idx[0]) / pd.Timedelta('365D')
year1_fit = year1_info['intercept'] + year1_info['slope'] * year1_years

# Year-2+ YoY median-rate line, anchored at the renormalizing factor
renorm = hybrid_info['renormalizing_factor_year2']
year2_idx = energy_normalized.loc[split:].index
year2_years = (year2_idx - year2_idx[0]) / pd.Timedelta('365D')
year2_fit = renorm * (1 + (rd2_hat / 100) * year2_years)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(energy_normalized.index, energy_normalized, '.', alpha=0.2,
        color='grey', label='noisy series')
ax.plot(year1_idx, year1_fit, color='C0', lw=2,
        label=f'year-1 OLS ({rd1_hat:+.2f} %/yr)')
ax.plot(year2_idx, year2_fit, color='C1', lw=2,
        label=f'years 2+ YoY ({rd2_hat:+.2f} %/yr)')
ax.axvline(split, color='red', ls='--', lw=1)
ax.set_ylabel('Normalized energy')
ax.legend(loc='lower left')
fig.autofmt_xdate()
plt.show()

## 4. Configuration knobs

- `year1_split` (default `1.0`): boundary in years. Set to a different value (e.g. `1.5` or `2.0`) if the qualitatively-different early-life regime is not exactly one year wide. Note that very small first-year windows (`< 1.0`) leave too little data for OLS to separate trend from seasonality.
- `recenter_year2` (default `True`): if `True`, the year-2+ rate is %/year of the start-of-year-2 capacity (the recommended interpretation). If `False`, the rate keeps the year-0 baseline of the input.
- `yoy_kwargs`: forwarded to `degradation_year_on_year` (e.g. `uncertainty_method='circular_block'`, `multi_yoy=True`).

The returned `calc_info` dict carries the full `(Rd_pct, Rd_CI, calc_info)` tuples from each underlying call under the keys `'year1'` and `'years2plus'`, so any downstream analysis or plotting that consumes a standard OLS or YoY result can be applied to either piece.

In [ ]:
# Example: wider year-1 window and circular-block bootstrap CIs on years 2+
rd1_alt, rd2_alt, info_alt = rdtools.degradation_hybrid_ols_yoy(
    energy_normalized,
    year1_split=1.5,
    yoy_kwargs={'uncertainty_method': 'circular_block', 'block_length': 90},
)
print(f'With year1_split=1.5:')
print(f'  first 1.5 yr (OLS):      {rd1_alt:+.3f} %/yr')
print(f'  remainder (YoY, CB CI):  {rd2_alt:+.3f} %/yr, '
      f'CI {info_alt["years2plus"][1]}')

## 5. Using the hybrid method through `TrendAnalysis`

The hybrid analysis is wired into the end-to-end `TrendAnalysis` workflow as `"hybrid_degradation"`. To run it alongside (or instead of) the default year-on-year analysis, list it in `analyses=` and pass any options through `hybrid_kwargs`.

The code cell below is illustrative — it assumes you have already constructed and configured a `TrendAnalysis` instance as shown in `TrendAnalysis_example.ipynb`.

In [ ]:
# rd_analysis.sensor_analysis(
#     analyses=['yoy_degradation', 'hybrid_degradation'],
#     hybrid_kwargs={'year1_split': 1.0, 'recenter_year2': True},
# )
#
# yoy_results    = rd_analysis.results['sensor']['yoy_degradation']
# hybrid_results = rd_analysis.results['sensor']['hybrid_degradation']
# print('YoY (whole series):       ', yoy_results['p50_rd'])
# print('Hybrid year 1:            ', hybrid_results['rd_pct_year1'])
# print('Hybrid years 2+:          ', hybrid_results['rd_pct_years2plus'])